# NLP Unit 2.0 — POS Tagging · Named Entity Recognition · Vectorization

This notebook covers three core NLP topics that build on Unit 1 preprocessing.

## Topics Covered:
### Part A — Part-of-Speech (POS) Tagging
1. What is POS Tagging? — Theory & Penn Treebank tagset
2. NLTK POS Tagger
3. spaCy POS Tagger
4. Kaggle news article: POS analysis & feature engineering

### Part B — Named Entity Recognition (NER)
5. What is NER? — Theory & entity types
6. NLTK NER (`ne_chunk`)
7. spaCy NER
8. Kaggle news article: entity extraction

### Part C — Language Model Vectorization
9. What is Vectorization? — BoW → TF-IDF → Word Embeddings
10. Bag of Words (BoW)
11. TF-IDF
12. Word Vectors (spaCy)
13. Kaggle news article: vectorization pipeline

---

## 1. What is Part-of-Speech (POS) Tagging?

**POS Tagging** (also called *grammatical tagging*) is the process of labelling each word in a sentence with its corresponding part of speech — such as noun, verb, adjective, adverb — based on both its definition and its context within the sentence.

### Why POS Tagging Matters

| Use Case | How POS Helps |
|----------|--------------|
| **Lemmatization** | Knowing POS → correct base form (`better` → `good` as ADV, not `better` as NOUN) |
| **Named Entity Recognition** | Proper nouns (NNP) signal potential entities |
| **Parsing & Syntax Trees** | Structural backbone of dependency/constituency parsing |
| **Machine Translation** | Correct word form depends on grammatical role |
| **Sentiment Analysis** | Adjectives and adverbs carry sentiment weight |
| **Information Extraction** | Verb–noun pairs reveal actions and subjects |

### How it Works
1. **Rule-based**: Hand-crafted regex/grammar rules (fast, brittle)
2. **Statistical (HMM / CRF)**: Trained on annotated corpora (NLTK default)
3. **Neural (Transformer)**: Context-aware embeddings (spaCy, stanza)

### Analogy
> Think of a sentence as a team. Each player (word) has a **position** (role).  
> A striker (verb) does the action; a defender (noun) is the subject/object;  
> an adjective describes a defender, an adverb describes the striker.

---

### Penn Treebank POS Tagset (used by NLTK)

| Tag | Part of Speech | Example |
|-----|---------------|---------|
| `NN` | Noun, singular | *dog, city* |
| `NNS` | Noun, plural | *dogs, cities* |
| `NNP` | Proper noun, singular | *London, Tesla* |
| `NNPS` | Proper noun, plural | *Vikings, Democrats* |
| `VB` | Verb
......
| `MD` | Modal | *can, will, should* |
| `WP` | Wh-pronoun | *who, what* |

> **spaCy** uses the Universal Dependencies (UD) tagset: `NOUN`, `VERB`, `ADJ`, `ADV`, `PRON`, `DET`, `ADP`, `CONJ`, `NUM`, `PUNCT`, `PROPN`, `AUX`, `PART`, `INTJ`, `SYM`, `X`

---

## 2. Setup & Imports

In [1]:
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from collections import Counter

# Download required NLTK data
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

print("✓ All libraries loaded successfully!")

✓ All libraries loaded successfully!


---

## 3. NLTK POS Tagging

NLTK's `pos_tag()` uses a **Maximum Entropy** model trained on the Penn Treebank corpus.  
It returns a list of `(word, tag)` tuples.

### Key Points:
- Requires tokenized input (`word_tokenize` first)
- Uses Penn Treebank tagset
- Faster but less context-aware than neural models
- The tagger is **positional** — the same word can get different tags based on neighbours

In [23]:
# ── NLTK POS Tagging ──────────────────────────────────────────────────────────

sentence = "The quick brown fox jumps over the lazy dog near the river."

# Step 1: Tokenize
tokens = word_tokenize(sentence)

# Step 2: POS tag
tagged = pos_tag(tokens)

print("Sentence:", sentence)
print("\n" + "="*60)
print(f"\n{'Token':<20} {'POS Tag':<12} Description")
print("-"*60)

tag_descriptions = {
    'DT': 'Determiner', 'JJ': 'Adjective', 'NN': 'Noun (singular)',
    'NNS': 'Noun (plural)', 'NNP': 'Proper Noun', 'VB': 'Verb (base)',
    'VBZ': 'Verb (3rd person)', 'VBD': 'Verb (past)', 'VBG': 'Verb (gerund)',
    'VBN': 'Verb (past participle)', 'RB': 'Adverb', 'IN': 'Preposition',
    'PRP': 'Personal Pronoun', 'CC': 'Conjunction', 'CD': 'Cardinal Number',
    'MD': 'Modal', '.': 'Punctuation', ',': 'Comma'
}

for word, tag in tagged:
    desc = tag_descriptions.get(tag, tag)
    print(f"  {word:<18} {tag:<12} {desc}")

Sentence: The quick brown fox jumps over the lazy dog near the river.


Token                POS Tag      Description
------------------------------------------------------------
  The                DT           Determiner
  quick              JJ           Adjective
  brown              NN           Noun (singular)
  fox                NN           Noun (singular)
  jumps              VBZ          Verb (3rd person)
  over               IN           Preposition
  the                DT           Determiner
  lazy               JJ           Adjective
  dog                NN           Noun (singular)
  near               IN           Preposition
  the                DT           Determiner
  river              NN           Noun (singular)
  .                  .            Punctuation


In [3]:
# ── Context Matters: Same word, different POS ─────────────────────────────────
# "run" can be a VERB or a NOUN depending on context

examples = [
    "I will run to the store.",          # run = VERB
    "Let's go for a morning run.",       # run = NOUN
    "She can book a flight.",            # book = VERB
    "I left the book on the table.",     # book = NOUN
]

print("Context-Dependent POS Tagging\n" + "="*50)
for sent in examples:
    tokens = word_tokenize(sent)
    tagged = pos_tag(tokens)
    print(f"\nSentence: {sent}")
    # Find the ambiguous word
    for word, tag in tagged:
        if word.lower() in ('run', 'book'):
            print(f"  → '{word}' tagged as: {tag} ({'Noun' if tag.startswith('N') else 'Verb' if tag.startswith('V') else tag})")

Context-Dependent POS Tagging

Sentence: I will run to the store.
  → 'run' tagged as: VB (Verb)

Sentence: Let's go for a morning run.
  → 'run' tagged as: NN (Noun)

Sentence: She can book a flight.
  → 'book' tagged as: NN (Noun)

Sentence: I left the book on the table.
  → 'book' tagged as: NN (Noun)


---

## 4. spaCy POS Tagging

spaCy uses a **neural CNN pipeline** trained on OntoNotes 5 corpus.  
It outputs two tag attributes per token:

| Attribute | Description | Example |
|-----------|-------------|---------|
| `token.pos_` | **Coarse-grained** Universal Dependencies tag | `VERB`, `NOUN` |
| `token.tag_` | **Fine-grained** Penn Treebank tag | `VBZ`, `NNS` |
| `token.dep_` | **Syntactic dependency** label | `nsubj`, `dobj` |

### Why spaCy > NLTK for POS?
- Considers **full sentence context** (not just bigrams)
- Outputs both coarse and fine-grained tags simultaneously
- Integrated into a full NLP pipeline (tokenizer → tagger → parser → NER)

In [4]:
# ── spaCy POS Tagging ─────────────────────────────────────────────────────────

sentence = "Apple is looking at buying a U.K. startup for $1 billion."

doc = nlp(sentence)

print(f"Sentence: {sentence}\n")
print(f"{'Token':<15} {'POS (coarse)':<14} {'Tag (fine)':<12} {'Dep':<10} {'Explanation'}")
print("-"*75)

for token in doc:
    print(f"  {token.text:<13} {token.pos_:<14} {token.tag_:<12} {token.dep_:<10} {spacy.explain(token.tag_)}")

Sentence: Apple is looking at buying a U.K. startup for $1 billion.

Token           POS (coarse)   Tag (fine)   Dep        Explanation
---------------------------------------------------------------------------
  Apple         PROPN          NNP          nsubj      noun, proper singular
  is            AUX            VBZ          aux        verb, 3rd person singular present
  looking       VERB           VBG          ROOT       verb, gerund or present participle
  at            ADP            IN           prep       conjunction, subordinating or preposition
  buying        VERB           VBG          pcomp      verb, gerund or present participle
  a             DET            DT           det        determiner
  U.K.          PROPN          NNP          dobj       noun, proper singular
  startup       NOUN           NN           advcl      noun, singular or mass
  for           ADP            IN           prep       conjunction, subordinating or preposition
  $             SYM        

In [5]:
# ── POS-aware Lemmatization (connecting back to Unit 1) ───────────────────────
# spaCy uses POS tag internally for correct lemmatization

print("POS Tag → Correct Lemma\n" + "="*45)

ambiguous = [
    "The studies were better than before.",
    "She studies hard every day.",
]

for sent in ambiguous:
    doc = nlp(sent)
    print(f"\nSentence: {sent}")
    for token in doc:
        if token.text.lower() in ('studies', 'better'):
            print(f"  '{token.text}' → pos={token.pos_}, lemma='{token.lemma_}'")

POS Tag → Correct Lemma

Sentence: The studies were better than before.
  'studies' → pos=NOUN, lemma='study'
  'better' → pos=ADJ, lemma='well'

Sentence: She studies hard every day.
  'studies' → pos=VERB, lemma='study'


---

## 5. Real-World Example — Kaggle News Article Analysis

We apply POS tagging to a **tech news article** (typical Kaggle NLP dataset format) to:
- Extract all **nouns** (key topics/entities)
- Extract all **verbs** (actions/events)
- Extract all **adjectives** (descriptive language / sentiment signals)
- Compute the **POS distribution** of the article

> This is the foundation of feature engineering for text classification tasks on Kaggle (e.g., AG News, BBC News datasets).

In [24]:
# ── Kaggle-style News Article ─────────────────────────────────────────────────
# Simulates a row from datasets like AG News / BBC News on Kaggle
# Category: Technology | Source: Kaggle NLP benchmark format

news_article = """
OpenAI has released GPT-4o, a powerful multimodal model that can process text, 
audio, and images simultaneously. The new model is significantly faster and 
cheaper than its predecessors, making advanced AI capabilities more accessible 
to developers worldwide. Tech analysts say the release marks a major shift in 
the competitive landscape, as Google and Meta are also racing to deploy similar 
foundation models. OpenAI CEO Sam Altman confirmed that GPT-4o will be available 
to free users of ChatGPT, while premium subscribers will receive higher usage limits. 
Industry experts believe this move will accelerate AI adoption across healthcare, 
education, and financial services sectors.
"""

print("NEWS ARTICLE:")
print("-" * 65)
print(news_article.strip())
print("-" * 65)

NEWS ARTICLE:
-----------------------------------------------------------------
OpenAI has released GPT-4o, a powerful multimodal model that can process text, 
audio, and images simultaneously. The new model is significantly faster and 
cheaper than its predecessors, making advanced AI capabilities more accessible 
to developers worldwide. Tech analysts say the release marks a major shift in 
the competitive landscape, as Google and Meta are also racing to deploy similar 
foundation models. OpenAI CEO Sam Altman confirmed that GPT-4o will be available 
to free users of ChatGPT, while premium subscribers will receive higher usage limits. 
Industry experts believe this move will accelerate AI adoption across healthcare, 
education, and financial services sectors.
-----------------------------------------------------------------


In [7]:
# ── spaCy POS Analysis of News Article ───────────────────────────────────────

doc = nlp(news_article)

nouns        = [t.text for t in doc if t.pos_ == 'NOUN']
proper_nouns = [t.text for t in doc if t.pos_ == 'PROPN']
verbs        = [t.lemma_ for t in doc if t.pos_ == 'VERB']
adjectives   = [t.text for t in doc if t.pos_ == 'ADJ']
adverbs      = [t.text for t in doc if t.pos_ == 'ADV']

print("=" * 65)
print("NOUNS (topics/objects):")
print(f"  {nouns}\n")
print("PROPER NOUNS (named entities):")
print(f"  {proper_nouns}\n")
print("VERBS (actions — lemmatized):")
print(f"  {verbs}\n")
print("ADJECTIVES (descriptors/sentiment signals):")
print(f"  {adjectives}\n")
print("ADVERBS (modifiers):")
print(f"  {adverbs}")
print("=" * 65)

NOUNS (topics/objects):
  ['model', 'text', 'images', 'model', 'predecessors', 'capabilities', 'developers', 'Tech', 'analysts', 'release', 'shift', 'landscape', 'foundation', 'models', 'users', 'subscribers', 'usage', 'limits', 'Industry', 'experts', 'move', 'adoption', 'healthcare', 'education', 'services', 'sectors']

PROPER NOUNS (named entities):
  ['OpenAI', 'GPT-4o', 'AI', 'Google', 'Meta', 'OpenAI', 'CEO', 'Sam', 'Altman', 'GPT-4o', 'ChatGPT', 'AI']

VERBS (actions — lemmatized):
  ['release', 'process', 'make', 'say', 'mark', 'race', 'deploy', 'confirm', 'receive', 'believe', 'accelerate']

ADJECTIVES (descriptors/sentiment signals):
  ['powerful', 'multimodal', 'audio', 'new', 'faster', 'cheaper', 'advanced', 'accessible', 'major', 'competitive', 'similar', 'available', 'free', 'premium', 'higher', 'financial']

ADVERBS (modifiers):
  ['simultaneously', 'significantly', 'more', 'worldwide', 'also']


In [8]:
# ── POS Distribution & Feature Extractor ─────────────────────────────────────

pos_counts = Counter(token.pos_ for token in doc if not token.is_space)
total = sum(pos_counts.values())

print("POS Tag Distribution")
print("=" * 48)
print(f"{'POS Tag':<10} {'Count':>5}  {'%':>6}  Bar")
print("-" * 48)
for pos, count in sorted(pos_counts.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {pos:<8} {count:>5}  {pct:>5.1f}%  {bar}")
print("-" * 48)
print(f"  {'TOTAL':<8} {total:>5}  100.0%")

print()

def extract_pos_features(text):
    doc = nlp(text)
    tokens = [t for t in doc if not t.is_space and not t.is_punct]
    total  = len(tokens)
    if total == 0:
        return {}
    pos_c = Counter(t.pos_ for t in tokens)
    return {
        "num_tokens":       total,
        "num_sentences":    len(list(doc.sents)),
        "noun_ratio":       round(pos_c.get('NOUN', 0)  / total, 3),
        "verb_ratio":       round(pos_c.get('VERB', 0)  / total, 3),
        "adj_ratio":        round(pos_c.get('ADJ', 0)   / total, 3),
        "adv_ratio":        round(pos_c.get('ADV', 0)   / total, 3),
        "propn_ratio":      round(pos_c.get('PROPN', 0) / total, 3),
        "top_nouns":        [t.text  for t in doc if t.pos_ == 'NOUN'][:5],
        "top_proper_nouns": [t.text  for t in doc if t.pos_ == 'PROPN'][:5],
        "top_verbs":        list(set(t.lemma_ for t in doc if t.pos_ == 'VERB'))[:5],
    }

print("Extracted POS Features:")
print("=" * 48)
for k, v in extract_pos_features(news_article).items():
    print(f"  {k:<20}: {v}")

POS Tag Distribution
POS Tag    Count       %  Bar
------------------------------------------------
  NOUN        26   23.0%  ███████████
  ADJ         16   14.2%  ███████
  PUNCT       13   11.5%  █████
  PROPN       12   10.6%  █████
  VERB        11    9.7%  ████
  AUX          8    7.1%  ███
  DET          6    5.3%  ██
  ADV          5    4.4%  ██
  ADP          5    4.4%  ██
  CCONJ        4    3.5%  █
  SCONJ        3    2.7%  █
  PRON         2    1.8%  
  PART         2    1.8%  
------------------------------------------------
  TOTAL      113  100.0%

Extracted POS Features:
  num_tokens          : 100
  num_sentences       : 5
  noun_ratio          : 0.26
  verb_ratio          : 0.11
  adj_ratio           : 0.16
  adv_ratio           : 0.05
  propn_ratio         : 0.12
  top_nouns           : ['model', 'text', 'images', 'model', 'predecessors']
  top_proper_nouns    : ['OpenAI', 'GPT-4o', 'AI', 'Google', 'Meta']
  top_verbs           : ['race', 'mark', 'process', 'release',

---

## Part A Summary — POS Tagging

| | NLTK | spaCy |
|--|------|-------|
| Tagset | Penn Treebank (fine-grained) | UD coarse + Penn fine |
| Speed | Fast | Slightly slower |
| Accuracy | Good | Better (neural) |
| Extra info | — | Dependency label, lemma, entity |
| Best for | Quick experiments | Production pipelines |

> **Key insight**: POS tags are the grammatical backbone — NER and vectorization both rely on accurate tagging upstream.

---
---

# Part B — Named Entity Recognition (NER)

---

## 6. What is Named Entity Recognition?

**Named Entity Recognition (NER)** is the task of locating and classifying **named entities** in text into predefined categories such as person names, organisations, locations, dates, and monetary values.

### Why NER Matters

| Use Case | How NER Helps |
|----------|--------------|
| **Information Extraction** | Pull who/what/where/when from unstructured text |
| **Knowledge Graphs** | Nodes = entities, edges = relationships |
| **Question Answering** | Identify the answer entity type before searching |
| **Recommendation** | Match user queries to product/company entities |
| **Finance/News analytics** | Track mentions of companies, people, stock tickers |
| **Healthcare** | Extract drug names, diseases, dosages from clinical notes |

### How it Works
1. **Rule-based**: Regex + gazetteers (lists of known names)
2. **Statistical (CRF)**: Learns entity boundaries from annotated corpora
3. **Neural (BiLSTM-CRF / Transformer)**: State-of-the-art; used by spaCy and Hugging Face

### NER Pipeline
```
Raw text → Tokenization → POS Tagging → NER Model → Entity spans + labels
```
POS tags (especially `PROPN`) are strong signals that feed into the NER model.

---

### spaCy Entity Types (OntoNotes 5)

| Label | Entity Type | Example |
|-------|------------|---------|
| `PERSON` | People, fictional | *Barack Obama, Sherlock Holmes* |
| `ORG` | Companies, agencies, institutions | *Google, NASA, WHO* |
| `GPE` | Countries, cities, states | *France, New York, California* |
| `LOC` | Non-GPE locations | *Amazon River, Mount Everest* |
| `PRODUCT` | Objects, vehicles, food | *iPhone, Tesla Model 3* |
| `DATE` | Absolute/relative dates | *2024, last Tuesday, Q3* |
| `TIME` | Times of day | *3pm, midnight* |
| `MONEY` | Monetary values | *$1 billion, €500* |
| `PERCENT` | Percentages | *95%, forty percent* |
| `CARDINAL` | Numerals not in other categories | *one, 42, dozen* |
| `ORDINAL` | "first", "second", etc. | *3rd, twenty-first* |
| `NORP` | Nationalities, political groups | *American, Democrats* |
| `FAC` | Facilities (buildings, airports) | *Heathrow, Pentagon* |
| `EVENT` | Named events | *World Cup, World War II* |
| `WORK_OF_ART` | Titles of works | *Mona Lisa, Harry Potter* |
| `LAW` | Legal documents | *GDPR, Constitution* |
| `LANGUAGE` | Named languages | *English, Python* |
| `QUANTITY` | Measurements | *5 km, 3 kg* |

---

## 7. NLTK NER — `ne_chunk`

NLTK uses a **Maximum Entropy** chunker to identify named entities.  
It requires POS-tagged tokens as input (NER builds directly on POS tagging).

### Limitations of NLTK NER:
- Only recognises: `PERSON`, `ORGANIZATION`, `GPE`, `LOCATION`, `FACILITY`, `GSP`
- Misses `DATE`, `MONEY`, `PRODUCT`, etc.
- Less accurate than spaCy on modern text

In [9]:
from nltk import ne_chunk, pos_tag, word_tokenize
from nltk.tree import Tree

nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('words', quiet=True)

sentence = "Elon Musk founded SpaceX in California and launched Falcon 9 in 2010."

# Step 1: Tokenize → Step 2: POS tag → Step 3: NE chunk
tokens = word_tokenize(sentence)
pos_tagged = pos_tag(tokens)
chunked = ne_chunk(pos_tagged)

print("Sentence:", sentence)
print("\nNLTK Named Entities:")
print("=" * 50)

for subtree in chunked:
    if isinstance(subtree, Tree):
        entity_text = " ".join(w for w, t in subtree.leaves())
        entity_label = subtree.label()
        print(f"  [{entity_label:<14}]  '{entity_text}'")

Sentence: Elon Musk founded SpaceX in California and launched Falcon 9 in 2010.

NLTK Named Entities:
  [PERSON        ]  'Elon'
  [PERSON        ]  'Musk'
  [ORGANIZATION  ]  'SpaceX'
  [GPE           ]  'California'


---

## 8. spaCy NER

spaCy's NER is part of its **neural pipeline** — it reads the full sentence context and uses transition-based parsing to identify entity spans.  
Access via `doc.ents` — each entity has `.text`, `.label_`, `.start_char`, `.end_char`.

In [10]:
# ── spaCy NER — basic ─────────────────────────────────────────────────────────

sentence = "Elon Musk founded SpaceX in California and launched Falcon 9 in 2010."

doc = nlp(sentence)

print("Sentence:", sentence)
print("\nspaCy Named Entities:")
print("=" * 60)
print(f"  {'Entity Text':<25} {'Label':<12} {'Description'}")
print("-" * 60)

for ent in doc.ents:
    print(f"  {ent.text:<25} {ent.label_:<12} {spacy.explain(ent.label_)}")

print()
print("spaCy finds DATE and CARDINAL — NLTK misses these entirely.")

Sentence: Elon Musk founded SpaceX in California and launched Falcon 9 in 2010.

spaCy Named Entities:
  Entity Text               Label        Description
------------------------------------------------------------
  Elon Musk                 PERSON       People, including fictional
  California                GPE          Countries, cities, states
  Falcon                    ORG          Companies, agencies, institutions, etc.
  2010                      DATE         Absolute or relative dates or periods

spaCy finds DATE and CARDINAL — NLTK misses these entirely.


In [11]:
# ── spaCy NER — inline highlighting ───────────────────────────────────────────
# Shows entity boundaries inside the original sentence

def highlight_entities(text):
    doc = nlp(text)
    result = []
    prev_end = 0
    for ent in doc.ents:
        result.append(text[prev_end:ent.start_char])
        result.append(f"[{ent.text} | {ent.label_}]")
        prev_end = ent.end_char
    result.append(text[prev_end:])
    return "".join(result)

sentences = [
    "Sam Altman, CEO of OpenAI, announced GPT-4o on Monday at the San Francisco headquarters.",
    "Apple's revenue exceeded $120 billion in Q4 2023, beating Wall Street forecasts.",
    "The United Nations met in Geneva to discuss the Paris Agreement on climate change.",
]

print("Inline Entity Highlighting\n" + "="*65)
for sent in sentences:
    print(f"\nOriginal : {sent}")
    print(f"Entities : {highlight_entities(sent)}")

Inline Entity Highlighting

Original : Sam Altman, CEO of OpenAI, announced GPT-4o on Monday at the San Francisco headquarters.
Entities : [Sam Altman | PERSON], CEO of [OpenAI | GPE], announced GPT-4o on [Monday | DATE] at the [San Francisco | GPE] headquarters.

Original : Apple's revenue exceeded $120 billion in Q4 2023, beating Wall Street forecasts.
Entities : [Apple | ORG]'s revenue exceeded [$120 billion | MONEY] in [Q4 2023 | DATE], beating Wall Street forecasts.

Original : The United Nations met in Geneva to discuss the Paris Agreement on climate change.
Entities : [The United Nations | ORG] met in [Geneva | GPE] to discuss [the Paris Agreement | EVENT] on climate change.


---

## 9. Real-World Example — NER on Kaggle News Article

Applying NER to the same tech news article to extract structured knowledge — who, what company, what product, when, how much.

In [12]:
# ── NER on Kaggle news article ────────────────────────────────────────────────

doc = nlp(news_article)

print("All Entities Found in News Article:")
print("=" * 60)
print(f"  {'Entity Text':<30} {'Label':<12} {'Description'}")
print("-" * 60)

for ent in doc.ents:
    print(f"  {ent.text.strip():<30} {ent.label_:<12} {spacy.explain(ent.label_)}")

All Entities Found in News Article:
  Entity Text                    Label        Description
------------------------------------------------------------
  OpenAI                         GPE          Countries, cities, states
  AI                             ORG          Companies, agencies, institutions, etc.
  Google                         ORG          Companies, agencies, institutions, etc.
  Meta                           ORG          Companies, agencies, institutions, etc.
  OpenAI                         ORG          Companies, agencies, institutions, etc.
  Sam Altman                     PERSON       People, including fictional
  AI                             GPE          Countries, cities, states


In [13]:
# ── NER Feature Extractor (Kaggle pipeline-ready) ─────────────────────────────

def extract_ner_features(text):
    """
    Returns structured entity groups + counts.
    Useful for Kaggle NLP tasks: news classification, relation extraction, etc.
    """
    doc = nlp(text)
    entity_groups = {}
    for ent in doc.ents:
        label = ent.label_
        entity_groups.setdefault(label, []).append(ent.text.strip())

    features = {
        "total_entities": len(doc.ents),
        "unique_labels":  list(entity_groups.keys()),
    }
    # Per-label lists
    for label in ['PERSON', 'ORG', 'GPE', 'PRODUCT', 'DATE', 'MONEY', 'CARDINAL']:
        features[label.lower()] = list(set(entity_groups.get(label, [])))

    return features

features = extract_ner_features(news_article)

print("NER Feature Dict (Kaggle-ready):")
print("=" * 50)
for k, v in features.items():
    print(f"  {k:<20}: {v}")

# Entity frequency distribution
ent_labels = [ent.label_ for ent in doc.ents]
label_counts = Counter(ent_labels)

print("\nEntity Label Distribution:")
print("=" * 40)
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"  {label:<12} {count:>3}  {bar}")

NER Feature Dict (Kaggle-ready):
  total_entities      : 7
  unique_labels       : ['GPE', 'ORG', 'PERSON']
  person              : ['Sam Altman']
  org                 : ['OpenAI', 'Google', 'Meta', 'AI']
  gpe                 : ['OpenAI', 'AI']
  product             : []
  date                : []
  money               : []
  cardinal            : []

Entity Label Distribution:
  ORG            4  ████
  GPE            2  ██
  PERSON         1  █


---

## Part B Summary — NER

| | NLTK `ne_chunk` | spaCy |
|--|----------------|-------|
| Entity types | 6 (PERSON, ORG, GPE…) | 18+ (DATE, MONEY, PRODUCT…) |
| Accuracy | Moderate | High (neural) |
| Multi-word entities | ✓ | ✓ |
| Character offsets | ✗ | ✓ (`ent.start_char`) |
| Best for | Quick prototyping | Production NLP |

> **Key insight**: NER turns unstructured text into **structured knowledge** — the foundation of information extraction, knowledge graphs, and question answering systems.

---
---

# Part C — Language Model Vectorization

---

## 10. What is Vectorization?

**Vectorization** converts raw text into **numerical vectors** that machine learning models can process.  
Words and sentences have no inherent numerical meaning — vectorization bridges language and mathematics.

### The Evolution of Text Vectorization

```
Bag of Words  →  TF-IDF  →  Word2Vec / GloVe  →  Contextual Embeddings (BERT)
  (sparse)       (sparse)      (dense, static)          (dense, contextual)
  simple         weighted      semantic meaning          state-of-the-art
```

### Comparison at a Glance

| Method | Type | Captures Semantics | Context-Aware | Dimension | Best For |
|--------|------|--------------------|--------------|-----------|----------|
| **Bag of Words (BoW)** | Sparse | ✗ | ✗ | Vocab size | Baseline, simple classifiers |
| **TF-IDF** | Sparse | Partial | ✗ | Vocab size | Document retrieval, classification |
| **Word2Vec / GloVe** | Dense | ✓ | ✗ (per word) | 100–300 | Similarity, analogy, embeddings |
| **spaCy vectors** | Dense | ✓ | ✗ (per word) | 96 (sm) / 300 (md) | Lightweight similarity |
| **BERT / GPT** | Dense | ✓ | ✓ (full context) | 768–1024 | SOTA NLP tasks |

### Key Terms
- **Sparse vector**: mostly zeros; length = vocabulary size (e.g., 50,000)
- **Dense vector**: small, all non-zero; fixed length (e.g., 300)
- **Embedding**: a learned dense representation capturing semantic meaning
- **Corpus**: collection of documents used to learn the vocabulary / embeddings

---

## 11. Bag of Words (BoW)

**BoW** represents a document as a vector of **word counts**, ignoring grammar and word order.

### How it works:
1. Build a **vocabulary** from the entire corpus
2. Represent each document as a vector: `[count(word1), count(word2), ...]`

### Pros & Cons:
| ✓ Pros | ✗ Cons |
|--------|--------|
| Simple to implement | Ignores word order ("dog bites man" = "man bites dog") |
| Interpretable | Sparse, high-dimensional |
| Fast | No semantic meaning |
| Works well as baseline | "good" and "great" treated as unrelated |

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# ── Bag of Words — manual demo ────────────────────────────────────────────────
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog are friends",
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

vocab = vectorizer.get_feature_names_out()
matrix = X.toarray()

print("Vocabulary:", list(vocab))
print(f"\nBoW Matrix  (shape: {matrix.shape} = {len(corpus)} docs × {len(vocab)} words)")
print("=" * 65)
print(f"{'':>5}", end="")
for w in vocab:
    print(f"{w:>8}", end="")
print()

for i, row in enumerate(matrix):
    print(f"Doc{i+1}:", end="")
    for v in row:
        print(f"{v:>8}", end="")
    print(f"  → '{corpus[i][:35]}'")

print()
print(f"Sparsity: {(matrix == 0).sum()} zeros out of {matrix.size} total values ({(matrix == 0).mean()*100:.0f}%)")

Vocabulary: ['and', 'are', 'cat', 'dog', 'friends', 'log', 'mat', 'on', 'sat', 'the']

BoW Matrix  (shape: (3, 10) = 3 docs × 10 words)
          and     are     cat     dog friends     log     mat      on     sat     the
Doc1:       0       0       1       0       0       0       1       1       1       2  → 'the cat sat on the mat'
Doc2:       0       0       0       1       0       1       0       1       1       2  → 'the dog sat on the log'
Doc3:       1       1       1       1       1       0       0       0       0       2  → 'the cat and the dog are friends'

Sparsity: 14 zeros out of 30 total values (47%)


---

## 12. TF-IDF (Term Frequency–Inverse Document Frequency)

**TF-IDF** weights each word by how *important* it is to a document relative to the corpus.  
Common words like "the" get low weight; rare informative words get high weight.

### Formula
```
TF(t, d)  = count of term t in document d / total terms in d
IDF(t)    = log( N / (1 + df(t)) )   where N = total docs, df(t) = docs containing t
TF-IDF    = TF × IDF
```

### Intuition:
- **High TF** → word appears often in *this* document (locally important)
- **High IDF** → word is *rare* across all documents (globally distinctive)
- **High TF-IDF** → word is important AND distinctive → strong feature

### Pros & Cons:
| ✓ Pros | ✗ Cons |
|--------|--------|
| Penalises common stop-words automatically | Still sparse & ignores word order |
| Better than raw BoW for classification | No semantic similarity |
| Industry standard for document retrieval | Long documents inflate TF |

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ── TF-IDF ────────────────────────────────────────────────────────────────────
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog are friends",
]

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)
vocab = tfidf.get_feature_names_out()
matrix = X_tfidf.toarray()

print("TF-IDF Matrix  (rounded to 2 dp)")
print("=" * 75)
print(f"{'':>5}", end="")
for w in vocab:
    print(f"{w:>9}", end="")
print()

for i, row in enumerate(matrix):
    print(f"Doc{i+1}:", end="")
    for v in row:
        print(f"{v:>9.2f}", end="")
    print(f"  → '{corpus[i][:35]}'")

print()
# Top TF-IDF words per document
print("Top TF-IDF words per document:")
print("=" * 45)
for i, row in enumerate(matrix):
    top_idx = np.argsort(row)[::-1][:3]
    top_words = [(vocab[j], round(row[j], 3)) for j in top_idx if row[j] > 0]
    print(f"  Doc{i+1}: {top_words}  ← '{corpus[i][:40]}'")

TF-IDF Matrix  (rounded to 2 dp)
           and      are      cat      dog  friends      log      mat       on      sat      the
Doc1:     0.00     0.00     0.37     0.00     0.00     0.00     0.49     0.37     0.37     0.58  → 'the cat sat on the mat'
Doc2:     0.00     0.00     0.00     0.37     0.00     0.49     0.00     0.37     0.37     0.58  → 'the dog sat on the log'
Doc3:     0.42     0.42     0.32     0.32     0.42     0.00     0.00     0.00     0.00     0.50  → 'the cat and the dog are friends'

Top TF-IDF words per document:
  Doc1: [('the', np.float64(0.581)), ('mat', np.float64(0.492)), ('sat', np.float64(0.374))]  ← 'the cat sat on the mat'
  Doc2: [('the', np.float64(0.581)), ('log', np.float64(0.492)), ('sat', np.float64(0.374))]  ← 'the dog sat on the log'
  Doc3: [('the', np.float64(0.501)), ('friends', np.float64(0.424)), ('are', np.float64(0.424))]  ← 'the cat and the dog are friends'


---

## 13. Word Vectors (Dense Embeddings) with spaCy

**Word vectors** map words to dense low-dimensional vectors where **similar words are geometrically close**.  
They are pre-trained on massive corpora (Wikipedia, Common Crawl) to capture semantic relationships.

### Famous Word Vector Properties:
```
king - man + woman ≈ queen
Paris - France + Germany ≈ Berlin
```

### spaCy Word Vectors:
- `en_core_web_sm` → 96-dim vectors (small)
- `en_core_web_md` → 300-dim vectors (GloVe trained, better similarity)
- Every `token.vector` → numpy array
- `doc.vector` → mean of all token vectors (sentence embedding)
- `token.similarity(other)` → cosine similarity [0, 1]

> **Note**: `en_core_web_sm` has limited vectors. For real similarity tasks use `en_core_web_md` or `en_core_web_lg`.

In [16]:
# ── Word Vectors with spaCy ───────────────────────────────────────────────────

words = ["king", "queen", "man", "woman", "Paris", "London", "cat", "dog"]

print("Word Vectors (first 8 dimensions shown):")
print("=" * 65)
print(f"  {'Word':<10} {'Vec dim':<10} {'First 8 values...'}")
print("-" * 65)

for word in words:
    token = nlp(word)[0]
    vec_preview = token.vector[:8].round(3)
    print(f"  {word:<10} {token.vector.shape[0]:<10} {vec_preview}")

Word Vectors (first 8 dimensions shown):
  Word       Vec dim    First 8 values...
-----------------------------------------------------------------
  king       96         [-1.057 -0.137 -0.081 -0.044 -0.128 -0.303  1.861  0.71 ]
  queen      96         [-0.804 -0.62   0.662 -0.391 -0.797 -0.536  0.379  1.341]
  man        96         [-0.37  -0.566 -0.223 -0.126  0.191 -0.69   0.84   0.806]
  woman      96         [-0.515 -0.465 -0.423 -0.172  0.429 -0.65   1.53   1.296]
  Paris      96         [-0.722 -0.141  0.883  0.8    0.458 -0.824  1.65   0.628]
  London     96         [-0.367 -1.036  1.545  0.878  0.314 -0.291  1.349  0.076]
  cat        96         [-0.841 -0.7    0.255 -0.055 -0.241 -0.155  1.249  0.865]
  dog        96         [-0.272 -0.938  0.536  0.445  0.383 -0.975  0.847  0.675]


In [17]:
# ── Cosine Similarity between sentences ───────────────────────────────────────

sentence_pairs = [
    ("I love machine learning",     "I enjoy deep learning"),        # similar
    ("The cat sat on the mat",      "A dog rested on a rug"),        # moderate
    ("The cat sat on the mat",      "OpenAI released a new model"),  # unrelated
]

print("Sentence Similarity (cosine via spaCy doc.vector):")
print("=" * 65)
for s1, s2 in sentence_pairs:
    doc1, doc2 = nlp(s1), nlp(s2)
    sim = doc1.similarity(doc2)
    bar = "█" * int(sim * 20)
    print(f"  {sim:.3f}  {bar}")
    print(f"         A: '{s1}'")
    print(f"         B: '{s2}'")
    print()

Sentence Similarity (cosine via spaCy doc.vector):
  0.715  ██████████████
         A: 'I love machine learning'
         B: 'I enjoy deep learning'

  0.742  ██████████████
         A: 'The cat sat on the mat'
         B: 'A dog rested on a rug'

  0.658  █████████████
         A: 'The cat sat on the mat'
         B: 'OpenAI released a new model'



/var/folders/0j/xl7l0bxj4055n123kt409zjm27pcyc/T/ipykernel_8302/2814776959.py:13: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  sim = doc1.similarity(doc2)


---

## 14. Real-World Example — Vectorizing the Kaggle News Article

Applying all three vectorization methods to the news article to compare their representations and show how they feed into a classifier.

In [18]:
# ── Kaggle corpus: 3 news articles from different categories ──────────────────

kaggle_corpus = {
    "tech":    news_article.strip(),
    "sports":  """
        The English Premier League season reached a thrilling conclusion as Manchester City 
        claimed their fourth consecutive title on the final matchday. Erling Haaland scored 
        a hat-trick to seal a 4-0 victory over West Ham, finishing the season with 52 goals. 
        Manager Pep Guardiola praised the squad's resilience and fitness throughout the campaign. 
        Arsenal finished second, their best Premier League result in over a decade.
    """.strip(),
    "finance": """
        The Federal Reserve held interest rates steady at 5.25% following its latest FOMC 
        meeting, citing persistent inflationary pressures and a robust labour market. 
        Fed Chair Jerome Powell signalled that rate cuts remain unlikely until inflation 
        consistently approaches the 2% target. US Treasury yields rose sharply in response, 
        with the 10-year note climbing to 4.8%. Stock markets fell, with the S&P 500 dropping 
        1.2% and the Nasdaq sliding 1.8%.
    """.strip(),
}

docs = list(kaggle_corpus.values())
labels = list(kaggle_corpus.keys())

print("Kaggle-style corpus (3 categories):")
for label, text in kaggle_corpus.items():
    print(f"\n  [{label.upper()}]: {text[:100]}...")

Kaggle-style corpus (3 categories):

  [TECH]: OpenAI has released GPT-4o, a powerful multimodal model that can process text, 
audio, and images si...

  [SPORTS]: The English Premier League season reached a thrilling conclusion as Manchester City 
        claimed...

  [FINANCE]: The Federal Reserve held interest rates steady at 5.25% following its latest FOMC 
        meeting, ...


In [19]:
# ── BoW on corpus ─────────────────────────────────────────────────────────────

bow_vec = CountVectorizer(stop_words='english', max_features=15)
X_bow = bow_vec.fit_transform(docs).toarray()

print("BoW — Top 15 features across corpus:")
print("=" * 55)
print(f"  {'Feature':<15}", end="")
for label in labels:
    print(f"  {label:<10}", end="")
print()
print("-" * 55)
for i, word in enumerate(bow_vec.get_feature_names_out()):
    print(f"  {word:<15}", end="")
    for j in range(len(docs)):
        print(f"  {X_bow[j, i]:<10}", end="")
    print()

print(f"\nBoW vector shape: {X_bow.shape}  ({X_bow.shape[0]} docs × {X_bow.shape[1]} features)")

BoW — Top 15 features across corpus:
  Feature          tech        sports      finance   
-------------------------------------------------------
  4o               2           0           0         
  ai               2           0           0         
  gpt              2           0           0         
  league           0           2           0         
  model            2           0           0         
  openai           2           0           0         
  praised          0           1           0         
  premier          0           2           0         
  racing           1           0           0         
  rate             0           0           1         
  rates            0           0           1         
  reached          0           1           0         
  release          1           0           0         
  released         1           0           0         
  season           0           2           0         

BoW vector shape: (3, 15)  (3 docs × 15 fe

In [20]:
# ── TF-IDF on corpus ──────────────────────────────────────────────────────────

tfidf_vec = TfidfVectorizer(stop_words='english', max_features=15)
X_tfidf = tfidf_vec.fit_transform(docs).toarray()

print("TF-IDF — Top 15 features (rounded):")
print("=" * 60)
print(f"  {'Feature':<18}", end="")
for label in labels:
    print(f"  {label:<10}", end="")
print()
print("-" * 60)
for i, word in enumerate(tfidf_vec.get_feature_names_out()):
    print(f"  {word:<18}", end="")
    for j in range(len(docs)):
        print(f"  {X_tfidf[j, i]:.3f}     ", end="")
    print()

print()
print("Top 5 TF-IDF keywords per article:")
print("=" * 50)
for i, label in enumerate(labels):
    top_idx = np.argsort(X_tfidf[i])[::-1][:5]
    top_kws = [(tfidf_vec.get_feature_names_out()[j], round(X_tfidf[i, j], 3)) for j in top_idx]
    print(f"  [{label.upper()}]: {top_kws}")

TF-IDF — Top 15 features (rounded):
  Feature             tech        sports      finance   
------------------------------------------------------------
  4o                  0.417       0.000       0.000     
  ai                  0.417       0.000       0.000     
  gpt                 0.417       0.000       0.000     
  league              0.000       0.535       0.000     
  model               0.417       0.000       0.000     
  openai              0.417       0.000       0.000     
  praised             0.000       0.267       0.000     
  premier             0.000       0.535       0.000     
  racing              0.209       0.000       0.000     
  rate                0.000       0.000       0.707     
  rates               0.000       0.000       0.707     
  reached             0.000       0.267       0.000     
  release             0.209       0.000       0.000     
  released            0.209       0.000       0.000     
  season              0.000       0.535       0.

In [21]:
# ── spaCy Doc Vectors (dense sentence embeddings) ─────────────────────────────

doc_vectors = np.array([nlp(text).vector for text in docs])

print("spaCy Doc Vectors (mean word embeddings):")
print("=" * 55)
print(f"  Vector shape per document: {doc_vectors[0].shape}")
print(f"  First 6 dims of each doc vector:")
for label, vec in zip(labels, doc_vectors):
    print(f"  [{label.upper():<8}]: {vec[:6].round(3)}")

print()
# Cross-document cosine similarity
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-9)

print("Cross-document cosine similarity (dense vectors):")
print("=" * 55)
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        sim = cosine_sim(doc_vectors[i], doc_vectors[j])
        print(f"  {labels[i]:<10} ↔ {labels[j]:<10}: {sim:.4f}")

print()
print("→ Lower similarity between tech/sports and tech/finance shows")
print("  embeddings capture topic differences, unlike BoW.")

spaCy Doc Vectors (mean word embeddings):
  Vector shape per document: (96,)
  First 6 dims of each doc vector:
  [TECH    ]: [-0.135 -0.18  -0.067 -0.123  0.08   0.017]
  [SPORTS  ]: [ 0.143 -0.368  0.035  0.116  0.099 -0.004]
  [FINANCE ]: [ 0.126 -0.218  0.002  0.115  0.207  0.005]

Cross-document cosine similarity (dense vectors):
  tech       ↔ sports    : 0.6732
  tech       ↔ finance   : 0.6972
  sports     ↔ finance   : 0.8707

→ Lower similarity between tech/sports and tech/finance shows
  embeddings capture topic differences, unlike BoW.


---

## Part C Summary — Vectorization

### Method Selection Guide

| Scenario | Recommended Method |
|----------|--------------------|
| Baseline text classifier (Naive Bayes, SVM) | **TF-IDF** |
| Keyword extraction / document search | **TF-IDF** |
| Semantic similarity / clustering | **Word Vectors (spaCy / GloVe)** |
| SOTA classification / generation | **BERT / GPT embeddings** |
| Tiny dataset, interpretability needed | **BoW** |

### Full Pipeline Summary (Unit 2)

```
Raw Text
   │
   ├─ Tokenize (Unit 1)
   ├─ Clean / Lemmatize (Unit 1)
   │
   ├─ POS Tag   → grammatical roles per token
   ├─ NER       → extract who / what / where / when
   └─ Vectorize → BoW / TF-IDF / embeddings → ML model
```

> These three techniques together cover the core feature engineering pipeline for almost every Kaggle NLP competition baseline.  
> **Unit 3** will apply all of these inside a full text classification pipeline.